In [0]:
%sql

create table customer_orders (
order_id integer,
customer_id integer,
order_date date,
order_amount integer
);

insert into customer_orders values(1,100,cast('2022-01-01' as date),2000),(2,200,cast('2022-01-01' as date),2500),(3,300,cast('2022-01-01' as date),2100)
,(4,100,cast('2022-01-02' as date),2000),(5,400,cast('2022-01-02' as date),2200),(6,500,cast('2022-01-02' as date),2700)
,(7,100,cast('2022-01-03' as date),3000),(8,400,cast('2022-01-03' as date),1000),(9,600,cast('2022-01-03' as date),3000);


select * from customer_orders;

In [0]:
%sql

create table if not exists workspace.sql_inputs.day3_customer_orders deep clone customer_orders;


In [0]:
df = spark.table("workspace.sql_inputs.day3_customer_orders")
df.show()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

first_date_df = df.withColumn("first_date", min("order_date").over(Window.partitionBy("customer_id")))

flags_df = first_date_df.withColumn("new_customer_flag", when(first_date_df.order_date == first_date_df.first_date, 1).otherwise(0)) \
.withColumn("repeat_customer_flag", when(first_date_df.order_date != first_date_df.first_date, 1).otherwise(0))

result_df = flags_df.groupBy(flags_df.order_date).agg(F.sum(flags_df.new_customer_flag).alias("new_customers"), F.sum(flags_df.repeat_customer_flag).alias("repeat_customers"))

result_df.show()